In [1]:
import pandas as pd
print(pd.__version__)

2.3.3


In [2]:
df=pd.read_csv("mabel.txt", header = None, on_bad_lines ="skip", encoding="utf8")
df

,0,1
0,05/12/19,1:42 pm - Messages to this chat and calls are...
1,05/12/19,1:42 pm - Mabel Infoziant: Hi this is Mabel w...
2,05/12/19,1:42 pm - Mabel Infoziant: What’s your full name
3,05/12/19,1:42 pm - AR❤: Ramisha Rani K
4,05/12/19,1:42 pm - Mabel Infoziant: Ok
5,05/12/19,1:42 pm - Mabel Infoziant: ramisharanik@gmail...
6,05/12/19,1:43 pm - Mabel Infoziant: Your email Id?
7,05/12/19,1:43 pm - AR❤: Yes Mam
8,05/12/19,1:43 pm - Mabel Infoziant: I will send 2 abst...
9,05/12/19,1:43 pm - AR❤: Yeah mam


In [3]:
df=df.drop(0)
df.columns=["Date","Chat"]
Message = df["Chat"].str.split("-",n=1,expand=True)
df["Time"]=Message[0]
Message1=Message[1].str.split(":",n=1,expand=True)
df["Name"]=Message1[0]
df["Chat"]=Message1[1]
df=df[["Date","Time","Name","Chat"]]
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


# 1. Flirt Prediction Model

#### A) Problem Statement
With the chat history, we have to identify whether the chat person flirted or not

In [ ]:
import numpy as np

In [ ]:
chat_data = df.copy()

In [ ]:
chat_data

In [ ]:
chat_data["Chat"] = chat_data["Chat"].fillna("").astype(str).str.strip()

In [ ]:
chat_data = chat_data[chat_data["Chat"] != ""].copy()

In [ ]:
chat_data = chat_data.reset_index(drop=True)

In [ ]:
chat_data["Message_ID"] = chat_data.index

In [ ]:
label_data = chat_data[["Message_ID", "Chat"]].copy()

In [ ]:
label_data["Flirt_Label"] = ""

In [ ]:
label_data.to_csv("whatsapp_messages_to_label.csv", index=False)

In [ ]:
display(label_data.head(10))

In [ ]:
print("Total messages to label:", len(label_data))
print("File created: whatsapp_messages_to_label.csv")

#### Counter Vector

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [11]:
data = pd.read_csv("whatsapp_messages_labeled.csv")
data

,Message_ID,Chat,Flirt_Label
0,0,Hi this is Mabel we just spoke,0
1,1,What’s your full name,0
2,2,Ramisha Rani K,0
3,3,Ok,0
4,4,ramisharanik@gmail.com,0
5,5,Your email Id?,0
6,6,Yes Mam,0
7,7,I will send 2 abstracts for u to start working,0
8,8,Yeah mam,0
9,9,Give me the list that u have too,0


In [12]:
data["Chat"] = data["Chat"].fillna("").astype(str)
data["Flirt_Label"] = pd.to_numeric(data["Flirt_Label"], errors="coerce")
data = data.dropna(subset=["Flirt_Label"])
data["Flirt_Label"] = data["Flirt_Label"].astype(int)
X = data["Chat"]
y = data["Flirt_Label"]

cv = CountVectorizer()
X_count = cv.fit_transform(X)

print("Count Vectorizer Shape:", X_count.shape)
print("Vocabulary Size:", len(cv.vocabulary_))

if y.nunique() == 2 and y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X_count, y, test_size=0.2,random_state=42, stratify=y)

    count_model = LogisticRegression(max_iter=1000)
    count_model.fit(X_train, y_train)

    y_pred = count_model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

    data["Count_Prediction"] = count_model.predict(X_count)
else:
    print("Classifier not trained: labels contain only one class.")
    print("Add genuine labelled examples from both classes to train.")

display(data.head())

Count Vectorizer Shape: (50, 189)
Vocabulary Size: 189
Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         8
           1       0.00      0.00      0.00         2

    accuracy                           0.80        10
   macro avg       0.40      0.50      0.44        10
weighted avg       0.64      0.80      0.71        10



,Message_ID,Chat,Flirt_Label,Count_Prediction
0,0,Hi this is Mabel we just spoke,0,0
1,1,What’s your full name,0,0
2,2,Ramisha Rani K,0,0
3,3,Ok,0,0
4,4,ramisharanik@gmail.com,0,0


#### TF-IDF Vector

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [14]:
data = pd.read_csv("whatsapp_messages_labeled.csv")
data

,Message_ID,Chat,Flirt_Label
0,0,Hi this is Mabel we just spoke,0
1,1,What’s your full name,0
2,2,Ramisha Rani K,0
3,3,Ok,0
4,4,ramisharanik@gmail.com,0
5,5,Your email Id?,0
6,6,Yes Mam,0
7,7,I will send 2 abstracts for u to start working,0
8,8,Yeah mam,0
9,9,Give me the list that u have too,0


In [15]:
data["Chat"] = data["Chat"].fillna("").astype(str)
data["Flirt_Label"] = pd.to_numeric(data["Flirt_Label"], errors="coerce")
data = data.dropna(subset=["Flirt_Label"])
data["Flirt_Label"] = data["Flirt_Label"].astype(int)

X = data["Chat"]
y = data["Flirt_Label"]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Shape:", X_tfidf.shape)
print("Vocabulary Size:", len(tfidf.vocabulary_))

if y.nunique() == 2 and y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2,random_state=42, stratify=y)

    tfidf_model = LogisticRegression(max_iter=1000)
    tfidf_model.fit(X_train, y_train)

    y_pred = tfidf_model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))
    data["TFIDF_Prediction"] = tfidf_model.predict(X_tfidf)
else:
    print("Classifier not trained: labels contain only one class.")
    print("Add genuine labelled examples from both classes to train.")

display(data.head())

TF-IDF Shape: (50, 189)
Vocabulary Size: 189
Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         8
           1       0.00      0.00      0.00         2

    accuracy                           0.80        10
   macro avg       0.40      0.50      0.44        10
weighted avg       0.64      0.80      0.71        10



,Message_ID,Chat,Flirt_Label,TFIDF_Prediction
0,0,Hi this is Mabel we just spoke,0,0
1,1,What’s your full name,0,0
2,2,Ramisha Rani K,0,0
3,3,Ok,0,0
4,4,ramisharanik@gmail.com,0,0


#### Hashing Vector

In [16]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [17]:
data = pd.read_csv("whatsapp_messages_labeled.csv")
data

,Message_ID,Chat,Flirt_Label
0,0,Hi this is Mabel we just spoke,0
1,1,What’s your full name,0
2,2,Ramisha Rani K,0
3,3,Ok,0
4,4,ramisharanik@gmail.com,0
5,5,Your email Id?,0
6,6,Yes Mam,0
7,7,I will send 2 abstracts for u to start working,0
8,8,Yeah mam,0
9,9,Give me the list that u have too,0


In [18]:
data["Chat"] = data["Chat"].fillna("").astype(str)
data["Flirt_Label"] = pd.to_numeric(data["Flirt_Label"], errors="coerce")
data = data.dropna(subset=["Flirt_Label"])
data["Flirt_Label"] = data["Flirt_Label"].astype(int)

X = data["Chat"]
y = data["Flirt_Label"]

hash_vectorizer = HashingVectorizer(n_features=2**12,alternate_sign=False)

X_hash = hash_vectorizer.transform(X)

print("Hashing Vectorizer Shape:", X_hash.shape)

if y.nunique() == 2 and y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X_hash, y, test_size=0.2,random_state=42, stratify=y)

    hash_model = LogisticRegression(max_iter=1000)
    hash_model.fit(X_train, y_train)

    y_pred = hash_model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

    data["Hashing_Prediction"] = hash_model.predict(X_hash)
else:
    print("Classifier not trained: labels contain only one class.")
    print("Add genuine labelled examples from both classes to train.")

display(data.head())

Hashing Vectorizer Shape: (50, 4096)
Accuracy: 0.8
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         8
           1       0.00      0.00      0.00         2

    accuracy                           0.80        10
   macro avg       0.40      0.50      0.44        10
weighted avg       0.64      0.80      0.71        10



,Message_ID,Chat,Flirt_Label,Hashing_Prediction
0,0,Hi this is Mabel we just spoke,0,0
1,1,What’s your full name,0,0
2,2,Ramisha Rani K,0,0
3,3,Ok,0,0
4,4,ramisharanik@gmail.com,0,0


In [19]:
print("Count Vectorizer features:", X_count.shape[1])
print("TF-IDF Vectorizer features:", X_tfidf.shape[1])
print("Hashing Vectorizer features:", X_hash.shape[1])

print("\nNumber of messages:", len(X))
print("Label distribution:")
print(y.value_counts())

Count Vectorizer features: 189
TF-IDF Vectorizer features: 189
Hashing Vectorizer features: 4096

Number of messages: 50
Label distribution:
Flirt_Label
0    42
1     8
Name: count, dtype: int64


In [20]:
if y.nunique() < 2:
    print("Model training cannot proceed.")
    print("All messages have the same label.")
    print("Classifiers need examples from both classes")
    print("(0 = Non-flirtatious, 1 = Flirtatious).")
else:
    print("Both classes are present.")
    print("You can proceed with classifier training.")

Both classes are present.
You can proceed with classifier training.


# 3) Talkative / Less talkative

In [ ]:
df["Name"].value_counts()

# 4) Most active day and time

In [ ]:
df["Date"].value_counts().head(1)

In [ ]:
df["Time"].value_counts().head(5)

# 5) Media count by each person

In [ ]:
media = df[df["Chat"].str.contains("file attached|Media omitted", na=False)]
media["Name"].value_counts()

# 6) Missed calls

In [ ]:
df[df["Chat"].str.contains("Missed", "missed", na=False)]